# Telco Churn - Business Value and Recommendations

In [ ]:
import json
import sys
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sys.path.append(str(Path("..").resolve()))

from src.explainability import compute_shap_values, get_top_shap_features
from src.models import load_processed_data

## Load model + data context

In [ ]:
x_train, x_test, y_train, y_test = load_processed_data("../data/processed")
best_model = joblib.load("../models/best_model.joblib")
best_threshold = float(joblib.load("../models/best_threshold.joblib"))

summary_path = Path("../models/explainability_summary.json")
if summary_path.exists():
    explainability_summary = json.loads(summary_path.read_text())
    top_features_prior = explainability_summary.get("top_features", [])
else:
    top_features_prior = []

y_proba = best_model.predict_proba(x_test)[:, 1]
pred_df = pd.DataFrame(
    {
        "row_id": x_test.index,
        "y_true": y_test.values,
        "churn_proba": y_proba,
    }
)

print("Test rows:", len(pred_df))
print("Best threshold from Phase 3-4:", round(best_threshold, 4))
print("Top features from Phase 5:", top_features_prior[:10])
print("debug:", pred_df["churn_proba"].describe().to_dict())

## Cost-benefit analysis and profit curve

In [ ]:
# Business assumptions (editable)
avg_clv_loss_if_churn = 1200.0
retention_offer_cost = 120.0
offer_success_rate = 0.35  # probability of preventing churn when intervention is targeted correctly

thresholds = np.linspace(0.05, 0.95, 91)
rows = []

for t in thresholds:
    predicted_positive = pred_df["churn_proba"] >= t
    tp = int(((predicted_positive) & (pred_df["y_true"] == 1)).sum())
    fp = int(((predicted_positive) & (pred_df["y_true"] == 0)).sum())
    interventions = int(predicted_positive.sum())

    retained_value = tp * offer_success_rate * avg_clv_loss_if_churn
    intervention_cost = interventions * retention_offer_cost
    net_profit = retained_value - intervention_cost

    rows.append(
        {
            "threshold": float(t),
            "tp": tp,
            "fp": fp,
            "interventions": interventions,
            "retained_value": retained_value,
            "intervention_cost": intervention_cost,
            "net_profit": net_profit,
        }
    )

profit_df = pd.DataFrame(rows)
best_profit_row = profit_df.loc[profit_df["net_profit"].idxmax()].copy()
best_profit_threshold = float(best_profit_row["threshold"])

print("Profit-optimal threshold:", round(best_profit_threshold, 4))
print("Net profit at optimum:", round(float(best_profit_row["net_profit"]), 2))
print("Interventions at optimum:", int(best_profit_row["interventions"]))

plt.figure(figsize=(10, 5))
sns.lineplot(data=profit_df, x="threshold", y="net_profit")
plt.axvline(best_profit_threshold, color="red", linestyle="--", label=f"Best={best_profit_threshold:.2f}")
plt.title("Profit Curve by Intervention Threshold")
plt.ylabel("Net Profit")
plt.legend()
plt.tight_layout()
plt.show()

## Segment high-risk customers by churn drivers

In [ ]:
shap_values, x_shap = compute_shap_values(best_model, x_test, max_samples=len(x_test))
top_features_current = get_top_shap_features(shap_values, list(x_shap.columns), top_n=10)

values = shap_values.values
if values.ndim == 3:
    values = values[:, :, 1]

shap_abs = np.abs(values)
row_top_idx = shap_abs.argmax(axis=1)
row_top_feature = [x_shap.columns[i] for i in row_top_idx]

seg_df = pd.DataFrame(
    {
        "row_id": x_shap.index,
        "churn_proba": best_model.predict_proba(x_shap)[:, 1],
        "top_driver": row_top_feature,
    }
)
high_risk = seg_df[seg_df["churn_proba"] >= best_profit_threshold].copy()

def map_segment(driver: str) -> str:
    if "MonthlyCharges" in driver or "charge_per_service" in driver:
        return "Price_sensitive_customers"
    if "Contract" in driver:
        return "Contract_lockin_risk"
    if "TechSupport" in driver or "OnlineSecurity" in driver:
        return "Under_supported_customers"
    if "tenure" in driver or "is_new_customer" in driver:
        return "New_or_short_tenure"
    return "Other_drivers"

high_risk["segment"] = high_risk["top_driver"].map(map_segment)
segment_summary = (
    high_risk.groupby("segment", as_index=False)
    .agg(customer_count=("row_id", "count"), avg_churn_proba=("churn_proba", "mean"))
    .sort_values("customer_count", ascending=False)
)
segment_summary

In [ ]:
plt.figure(figsize=(10, 5))
sns.barplot(data=segment_summary, x="segment", y="customer_count")
plt.title("High-Risk Customer Segment Sizes")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 5))
sns.barplot(data=segment_summary, x="segment", y="avg_churn_proba")
plt.title("Average Churn Probability by Segment")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

## Actionable recommendations + estimated ROI

In [ ]:
recommendation_map = {
    "Price_sensitive_customers": "Offer targeted pricing relief or plan right-sizing",
    "Contract_lockin_risk": "Offer 12-month migration incentive with benefits",
    "Under_supported_customers": "Proactive support onboarding and security bundle",
    "New_or_short_tenure": "First-90-day retention journey and success outreach",
    "Other_drivers": "Case-by-case retention outreach",
}

roi_rows = []
for _, row in segment_summary.iterrows():
    seg = row["segment"]
    n = int(row["customer_count"])
    p = float(row["avg_churn_proba"])
    expected_tp = n * p
    retained_value = expected_tp * offer_success_rate * avg_clv_loss_if_churn
    cost = n * retention_offer_cost
    net = retained_value - cost
    roi_rows.append(
        {
            "segment": seg,
            "customers": n,
            "avg_churn_proba": p,
            "recommended_action": recommendation_map.get(seg, "Case-by-case retention outreach"),
            "expected_net_value": net,
        }
    )

roi_df = pd.DataFrame(roi_rows).sort_values("expected_net_value", ascending=False)
roi_df

## Quick summary dashboard

In [ ]:
kpi = {
    "test_customers": int(len(y_test)),
    "model_threshold_phase34": float(best_threshold),
    "profit_optimal_threshold": float(best_profit_threshold),
    "max_net_profit": float(best_profit_row["net_profit"]),
    "high_risk_customers_at_optimal_threshold": int(best_profit_row["interventions"]),
    "top_churn_drivers": top_features_current[:3],
    "top_recommendation_segments": roi_df.head(3)["segment"].tolist(),
}

print("Phase 6 Dashboard KPIs")
for key, value in kpi.items():
    print(f"- {key}: {value}")

models_dir = Path("../models")
models_dir.mkdir(parents=True, exist_ok=True)
with open(models_dir / "business_value_summary.json", "w", encoding="utf-8") as f:
    json.dump(
        {
            "assumptions": {
                "avg_clv_loss_if_churn": avg_clv_loss_if_churn,
                "retention_offer_cost": retention_offer_cost,
                "offer_success_rate": offer_success_rate,
            },
            "kpi": kpi,
            "segment_summary": segment_summary.to_dict(orient="records"),
            "roi_summary": roi_df.to_dict(orient="records"),
        },
        f,
        indent=2,
    )

profit_df.to_csv(models_dir / "profit_curve.csv", index=False)
roi_df.to_csv(models_dir / "segment_recommendations.csv", index=False)

print("\nSaved:")
print("-", models_dir / "business_value_summary.json")
print("-", models_dir / "profit_curve.csv")
print("-", models_dir / "segment_recommendations.csv")